In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Read CSV file 
df = pd.read_csv("SIS_Faculty-List.csv", encoding="ISO-8859-1")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.isnull().sum().sum()

In [ ]:
# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
duplicate_rows

In [ ]:
# Basic statistics for numerical columns to help identify outliers
numerical_stats = df.describe()
numerical_stats

In [ ]:
# Before cleansing
total_values = df.size
total_missing_values_before = df.isnull().sum().sum()
percentage_missing_before = (total_missing_values_before / total_values) * 100
print(percentage_missing_before)

In [ ]:
# Calculate missing values before cleansing
missing_before = df.isna().sum()

In [ ]:
# Drop columns
columns_to_drop = ["LWD", "Highest Qualification",
                   "All Qualifications from Profile",
                   "Courses Taught- Term 201510",
                   "MAJOR TEACHING FIELD",
                   "DOCUMENT OTHER PROFESSIONAL CERTIFICATION CRITIERA Five Years Work Experience Teaching Excellence Professional Certifications",
                   "Type",
                   "Divison",
                   "Location",
                   "Name",
                   "Reports To",
                   "Major",
                   "University"]
df_cleaned = df.drop(columns=columns_to_drop)

In [ ]:
df_cleaned.to_csv("SIS_Faculty-List_cleaned.csv")

In [ ]:
df_cleaned.head()

In [ ]:
# Rename columns
columns_to_renamed = {
    'Join\nDate': 'JoinDate',
    'Highest\nQualification\nLevel': 'Qualification'
}
df_cleaned = df_cleaned.rename(columns=columns_to_renamed)

In [ ]:
df_cleaned.head()

In [ ]:
(df['ID'] == '0').sum()

In [ ]:
# Filter ID column which includes a non-zero and null
df_cleaned = df_cleaned.loc[(df_cleaned['ID'] != '0') & (df_cleaned['ID'].notnull())]

In [ ]:
df_cleaned.head()

In [ ]:
id_data_constraint = 'LT'
non_LT_ID_count = len(df_cleaned) - df_cleaned['ID'].str.startswith(id_data_constraint).sum()
non_LT_ID_count

In [ ]:
mask = df_cleaned['ID'].str.startswith(id_data_constraint)
df_cleaned = df_cleaned[mask]

In [ ]:
df_cleaned.isnull().sum()

In [ ]:
df_cleaned.isna().sum()

In [ ]:
# Convert FA Grade to 0.0
df_cleaned['Grade'] = df_cleaned['Grade'].replace('FA', 0.0)

In [ ]:
df_cleaned['Qualification'].value_counts()

In [ ]:
df_cleaned.Qualification.isna().sum()

In [ ]:
df_cleaned = df_cleaned.dropna(subset=['Qualification'])

In [ ]:
df_cleaned.Qualification.isna().sum()

In [ ]:
df_cleaned['Qualification'] = df_cleaned['Qualification'].str.lower()
valid_qualifications = ['masters', 'doctorate', 'ph.d', 'bachelor']
df_cleaned = df_cleaned[df_cleaned['Qualification'].isin(valid_qualifications)]

In [ ]:
df_cleaned.head()

In [ ]:
qualification_mapping = {"bachelor": 1, "masters": 2, "doctorate": 3, "ph.d": 4}
df_cleaned['Qualification'] = df_cleaned['Qualification'].map(qualification_mapping)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.Title.value_counts()

In [ ]:
df_cleaned.loc[(df_cleaned['Title'] != 'Faculty - Trainee') & (
        df_cleaned['Title'] != 'Graduate Trainee'), 'Title'] = 'Faculty - Business'
df_cleaned['Title'].value_counts()

In [ ]:
title_mapping_dict = {
    'Faculty - Business': 'Faculty-Business',
    'Faculty - Trainee': 'Faculty-Trainee',
    'Graduate Trainee': 'Faculty-Trainee',
}
df_cleaned['Title'] = df_cleaned['Title'].replace(title_mapping_dict)

In [ ]:
df_cleaned.Title.value_counts()

In [ ]:
title_mapping = {"Faculty-Business": 1, "Faculty-Trainee": 2}
df_cleaned['Title'] = df_cleaned['Title'].map(title_mapping)

In [ ]:
df_cleaned.Title.value_counts()

In [ ]:
df_cleaned.Criteria.value_counts()

In [ ]:
# Create a dataframe with each category separated
criteria = df_cleaned['Criteria'].str.get_dummies('/')
df_cleaned = df_cleaned.drop(columns=['Criteria'])

# Join with the original dataframe
df_cleaned = df_cleaned.join(criteria)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned['JoinDate'] = pd.to_datetime(df_cleaned['JoinDate'], format='%d-%b-%y')

In [ ]:
df_cleaned.isna().sum()

In [ ]:
# After cleansing
total_missing_values_after = df_cleaned.isnull().sum().sum()
percentage_missing_after = (total_missing_values_after / total_values) * 100
percentage_missing_after

In [ ]:
# Calculate missing values after cleansing
missing_after = df_cleaned.isna().sum()

In [ ]:
# Create a combined DataFrame
missing_df = pd.concat([missing_before, missing_after], axis=1)
missing_df.columns = ['before', 'after']

In [ ]:
# Plotting
missing_df.plot(kind='bar', figsize=(5,5))
plt.ylabel('Missing Values')
plt.title('Missing Values Before and After Data Cleaning')
plt.show()
# Save figure
plt.savefig('missing_values.png', dpi=600, bbox_inches='tight')
plt.close()

In [ ]:
# Calculate the '0' values in 'ID' column before cleaning
ID_zero_counts_before = (df['ID'] == '0').sum()

# Calculate the '0' values in 'ID' column after cleaning
ID_zero_counts_after = (df_cleaned['ID'] == '0').sum()

In [ ]:
# Create a DataFrame to plot
ID_zero_counts_df = pd.DataFrame({'Before Cleaning': [ID_zero_counts_before], 'After Cleaning': [ID_zero_counts_after]})
ID_zero_counts_df = ID_zero_counts_df.transpose()

In [ ]:
# Plotting
ID_zero_counts_df.plot(kind='bar', legend = False)
plt.ylabel('Number of ID == 0')
plt.title('Comparison of ID==0 Before and After Data Cleaning')

In [ ]:
# Save figure
plt.tight_layout()
plt.savefig('ID_zero_counts_comparison.png', dpi=600, bbox_inches='tight')

plt.close()

In [ ]:
# Number of rows before cleaning
rows_before = df.shape[0]

# Number of rows after cleaning
rows_after = df_cleaned.shape[0]

In [ ]:
# Create a DataFrame for plotting
rows_df = pd.DataFrame({'Before Cleaning': [rows_before], 'After Cleaning': [rows_after]})
rows_df = rows_df.transpose()

In [ ]:
# Plotting
rows_df.plot(kind='bar', legend = False)
plt.ylabel('Number of Rows')
plt.title('Comparison of Number of Rows Before and After Data Cleaning')

In [ ]:
# Save figure
plt.tight_layout()
plt.savefig('rows_comparison.png', dpi=600, bbox_inches='tight')